In [6]:
from ortools.linear_solver import pywraplp
import plotly.express as px
from datetime import datetime, timedelta

def main():
    solver = pywraplp.Solver.CreateSolver("SCIP")

    # ====================== DATA SETUP ======================
    num_days = 7
    num_shifts = 3  # Morning, Afternoon, Evening
    manager_ids = [0, 1, 2, 3]  # 0 = Store Manager
    cashier_ids = list(range(9))
    stocker_ids = [0, 1]

    manager_shift_hours = 8
    cashier_shift_hours = 5
    stocker_shift_hours = 4

    manager_max_days = 5
    cashier_night_shift = 2
    no_night_cashiers = [0, 1, 2, 3, 4]

    friday_index = 4
    wednesday_index = 2
    thursday_index = 3

    stocker_friday_required = [1, 1]
    stocker_unavailable = {
        0: [wednesday_index, thursday_index],
        1: []
    }

    max_weekly_hours = 240

    # ====================== VARIABLES ======================
    m = {(i, d, s): solver.BoolVar(f"m_{i}_{d}_{s}")
         for i in manager_ids for d in range(num_days) for s in range(2)}
    c = {(i, d, s): solver.BoolVar(f"c_{i}_{d}_{s}")
         for i in cashier_ids for d in range(num_days) for s in range(num_shifts)}
    st = {(i, d): solver.BoolVar(f"st_{i}_{d}")
          for i in stocker_ids for d in range(num_days)}

    # ====================== CONSTRAINTS ======================
    for d in range(num_days):
        for s in range(2):
            solver.Add(sum(m[i, d, s] for i in manager_ids) == 1)

    for i in manager_ids:
        solver.Add(sum(m[i, d, s] for d in range(num_days) for s in range(2)) <= manager_max_days)

    for d in [5, 6]:
        for s in range(2):
            solver.Add(m[0, d, s] == 0)

    for d in range(5):
        solver.Add(sum(m[0, d, s] for s in range(2)) >= 1)

    for i in manager_ids:
        for d in range(num_days):
            solver.Add(sum(m[i, d, s] for s in range(2)) <= 1)

    for d in range(num_days):
        for s in range(num_shifts):
            solver.Add(sum(c[i, d, s] for i in cashier_ids) == 1)

    for i in cashier_ids:
        for d in range(num_days):
            solver.Add(sum(c[i, d, s] for s in range(num_shifts)) <= 1)

    for i in no_night_cashiers:
        for d in range(num_days):
            solver.Add(c[i, d, cashier_night_shift] == 0)

    for i in stocker_ids:
        solver.Add(st[i, friday_index] == stocker_friday_required[i])

    for i in stocker_unavailable:
        for d in stocker_unavailable[i]:
            solver.Add(st[i, d] == 0)

    manager_hours = sum(m[i, d, s] * manager_shift_hours for i in manager_ids for d in range(num_days) for s in range(2))
    cashier_hours = sum(c[i, d, s] * cashier_shift_hours for i in cashier_ids for d in range(num_days) for s in range(num_shifts))
    stocker_hours = sum(st[i, d] * stocker_shift_hours for i in stocker_ids for d in range(num_days))

    total_hours = manager_hours + cashier_hours + stocker_hours
    solver.Add(total_hours <= max_weekly_hours)

    max_cashier_shifts = solver.IntVar(0, num_days * num_shifts, 'max_cashier_shifts')
    min_cashier_shifts = solver.IntVar(0, num_days * num_shifts, 'min_cashier_shifts')

    for i in cashier_ids:
        total_shifts = sum(c[i, d, s] for d in range(num_days) for s in range(num_shifts))
        solver.Add(total_shifts <= max_cashier_shifts)
        solver.Add(total_shifts >= min_cashier_shifts)

    stocker_extra_shifts = [
        sum(st[i, d] for d in range(num_days) if d != friday_index and d not in stocker_unavailable[i])
        for i in stocker_ids
    ]

    max_stocker_extra = solver.IntVar(0, num_days, 'max_stocker_extra')
    min_stocker_extra = solver.IntVar(0, num_days, 'min_stocker_extra')

    for shifts in stocker_extra_shifts:
        solver.Add(shifts <= max_stocker_extra)
        solver.Add(shifts >= min_stocker_extra)

    solver.Maximize(
        total_hours
        - 5 * (max_cashier_shifts - min_cashier_shifts)
        - 3 * (max_stocker_extra - min_stocker_extra)
    )

    # ====================== SOLVE & PRINT ======================
    status = solver.Solve()

    if status == pywraplp.Solver.OPTIMAL:
        print("\nOptimal schedule found!")
        print(f"Total hours used: {total_hours.solution_value()}/{max_weekly_hours}")
        print(f"Utilization: {total_hours.solution_value()/max_weekly_hours:.2%}")

        days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
        shift_names_m = {0: "Opening (8am-4pm)", 1: "Closing (2pm-10pm)"}
        shift_names_c = {0: "Morning (8am-1pm)", 1: "Afternoon (1pm-6pm)", 2: "Evening (6pm-10pm)"}
        manager_names = {0: "Store Manager", 1: "Asst Manager 1", 2: "Asst Manager 2", 3: "Asst Manager 3"}

        print("\n===== MANAGER SCHEDULE =====")
        for d in range(num_days):
            print(f"\n{days[d]}:")
            for s in range(2):
                for i in manager_ids:
                    if m[i, d, s].solution_value() > 0.5:
                        print(f"  {shift_names_m[s]}: {manager_names[i]}")

        print("\n===== CASHIER SCHEDULE =====")
        for i in cashier_ids:
            print(f"\nCashier {i+1} (No nights)" if i in no_night_cashiers else f"\nCashier {i+1}:")
            shifts_assigned = 0
            for d in range(num_days):
                for s in range(num_shifts):
                    if c[i, d, s].solution_value() > 0.5:
                        print(f"  {days[d]}: {shift_names_c[s]}")
                        shifts_assigned += 1
            print(f"  Total shifts: {shifts_assigned}")

        print("\n===== STOCKER SCHEDULE =====")
        for i in stocker_ids:
            print(f"\nStocker {i+1}:")
            days_assigned = 0
            for d in range(num_days):
                if st[i, d].solution_value() > 0.5:
                    print(f"  {days[d]}: {stocker_shift_hours} hour shift")
                    days_assigned += 1
            print(f"  Total shifts: {days_assigned} (Mandatory Friday + {days_assigned - 1} extra)")

        # ========== Use the code below to plot the GANTT CHART ==========

        """print("\nGenerating Gantt-style shift visualization...")

        base_date = datetime(2025, 6, 9, 0, 0)

        gantt_data = []

        for i in manager_ids:
            for d in range(num_days):
                for s in range(2):
                    if m[i, d, s].solution_value() > 0.5:
                        start_hour = 8 if s == 0 else 14
                        end_hour = 16 if s == 0 else 22
                        task_date = base_date + timedelta(days=d)
                        gantt_data.append({
                            "Task": f"Manager {i}",
                            "Start": task_date + timedelta(hours=start_hour),
                            "Finish": task_date + timedelta(hours=end_hour),
                            "Role": "Manager"
                        })

        for i in cashier_ids:
            for d in range(num_days):
                for s in range(num_shifts):
                    if c[i, d, s].solution_value() > 0.5:
                        shift_start = [8, 13, 18][s]
                        shift_end = [13, 18, 22][s]
                        task_date = base_date + timedelta(days=d)
                        gantt_data.append({
                            "Task": f"Cashier {i}",
                            "Start": task_date + timedelta(hours=shift_start),
                            "Finish": task_date + timedelta(hours=shift_end),
                            "Role": "Cashier"
                        })

        for i in stocker_ids:
            for d in range(num_days):
                if st[i, d].solution_value() > 0.5:
                    task_date = base_date + timedelta(days=d)
                    gantt_data.append({
                        "Task": f"Stocker {i}",
                        "Start": task_date + timedelta(hours=9),
                        "Finish": task_date + timedelta(hours=13),
                        "Role": "Stocker"
                    })

        fig = px.timeline(
            gantt_data,
            x_start="Start",
            x_end="Finish",
            y="Task",
            color="Role",
            title="Employee Shift Schedule (Gantt Chart)",
        )
        fig.update_yaxes(autorange="reversed")
        fig.show()
        fig.update_yaxes(autorange="reversed")
        fig.update_xaxes(
    tickformat="%a %H",
    title="Day and Time"
)

    else:
        print("No optimal solution found. Status:", status)"""

if __name__ == "__main__":
    main()



Optimal schedule found!
Total hours used: 237.0/240
Utilization: 98.75%

===== MANAGER SCHEDULE =====

Mon:
  Opening (8am-4pm): Store Manager
  Closing (2pm-10pm): Asst Manager 3

Tue:
  Opening (8am-4pm): Store Manager
  Closing (2pm-10pm): Asst Manager 3

Wed:
  Opening (8am-4pm): Asst Manager 2
  Closing (2pm-10pm): Store Manager

Thu:
  Opening (8am-4pm): Asst Manager 2
  Closing (2pm-10pm): Store Manager

Fri:
  Opening (8am-4pm): Store Manager
  Closing (2pm-10pm): Asst Manager 1

Sat:
  Opening (8am-4pm): Asst Manager 3
  Closing (2pm-10pm): Asst Manager 1

Sun:
  Opening (8am-4pm): Asst Manager 3
  Closing (2pm-10pm): Asst Manager 1

===== CASHIER SCHEDULE =====

Cashier 1 (No nights)
  Wed: Morning (8am-1pm)
  Fri: Morning (8am-1pm)
  Total shifts: 2

Cashier 2 (No nights)
  Wed: Afternoon (1pm-6pm)
  Thu: Afternoon (1pm-6pm)
  Total shifts: 2

Cashier 3 (No nights)
  Sat: Afternoon (1pm-6pm)
  Sun: Afternoon (1pm-6pm)
  Total shifts: 2

Cashier 4 (No nights)
  Tue: Afternoo